# Test Data Masking for Fair Evaluation

This notebook documents the motivation and implementation of test data masking in BiGG training. By excluding test-set information from the training loss, we prevent data leakage when evaluating synthetic graphs on downstream benchmarks.

**Current scope:** Label masking for anomaly detection (`--mask_test_labels`).
**Planned:** Edge masking for link prediction (future work).

**Overview:**
1. The data leakage problem
2. Label masking: what it does (and does not do)
3. Implementation details
4. Scope and limitations
5. Usage

## 1. The Data Leakage Problem

In the anomaly detection benchmark, a GNN is trained on the synthetic graph and tested on the **original graph's test nodes**. The evaluation question is: *does the synthetic graph capture enough of the real data's structure, features, and label patterns for a GNN to generalise to unseen real data?*

However, BiGG trains on the **entire** original graph — including the labels of nodes that will later be used as the test set. The autoregressive label predictor is optimised to reproduce these test labels via cross-entropy loss. If the model memorises them, the synthetic graph's labels in the corresponding region of the graph are directly informed by the test signal.

This creates a subtle leakage path:

```
Original graph (all labels) → BiGG training (label loss on all nodes)
    → Synthetic graph (labels informed by test nodes)
        → GNN trained on synthetic → tested on original test nodes
```

The GNN trained on synthetic data benefits from label patterns that were "seen" during generation. This can inflate anomaly detection scores, making the synthetic data appear more useful than it actually is.

Note that this is **not** the same as the GNN seeing test labels directly — the leakage is indirect, mediated through the generative model. But the label predictor's loss explicitly optimises for reproducing those labels, which is the strongest form of indirect leakage.

## 2. Label Masking: What It Does (and Does Not Do)

The `--mask_test_labels` flag builds a boolean mask from the original graph's train/val/test split (split 0) and applies it to the label loss during BiGG training. Specifically:

**What is masked:**
- The **label cross-entropy loss** is computed only on train + val nodes. Test node labels do not contribute to the loss or its gradients.
- The **calibration pass** (which estimates loss magnitudes for dynamic weight normalisation) also uses the mask, so loss weights are calibrated on the same subset.

**What is NOT masked:**
- **Continuous feature loss** (`ll_cont`) is still computed on all nodes. Features are not part of the test signal in anomaly detection — the leakage concern is specifically about labels.
- **Structural loss** is unchanged — all edges contribute to topology learning.
- **Teacher forcing** still uses all labels. During the autoregressive forward pass, the ground-truth label of every node (including test nodes) is embedded and fed into the LSTM for state updates. Only the label *prediction loss* is masked.

### Why still use test labels for teacher forcing?

The autoregressive LSTM builds up a hidden state trajectory node-by-node. Each node's label embedding feeds into the state update that conditions subsequent predictions. If we replaced test node labels with a dummy value (e.g., zero or a uniform embedding), the hidden state trajectory would diverge from what the model sees during generation (where it uses its own predicted labels). This would degrade the quality of structural and feature predictions for all subsequent nodes in the autoregressive ordering.

By keeping teacher forcing intact and only masking the loss, we ensure:
1. The LSTM hidden state trajectory remains faithful to the true graph.
2. The label predictor is not optimised to reproduce test labels.
3. Structure and feature generation quality are unaffected.

In [ ]:
# Illustration: how the mask is constructed and applied

import torch
import torch.nn.functional as F

# Simulate a graph with 10 nodes, 2 classes
torch.manual_seed(42)
num_nodes = 10

# Simulated train/val/test split (split 0)
train_mask = torch.tensor([1, 1, 1, 0, 0, 0, 0, 0, 0, 0], dtype=torch.bool)
val_mask   = torch.tensor([0, 0, 0, 1, 1, 0, 0, 0, 0, 0], dtype=torch.bool)
test_mask  = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=torch.bool)

# label_mask: True = include in label loss
label_mask = train_mask | val_mask  # nodes 0-4

print(f"Train nodes: {train_mask.nonzero().squeeze().tolist()}")
print(f"Val nodes:   {val_mask.nonzero().squeeze().tolist()}")
print(f"Test nodes:  {test_mask.nonzero().squeeze().tolist()}")
print(f"Label mask:  {label_mask.tolist()}")
print(f"  → {label_mask.sum()} nodes contribute to label loss, {(~label_mask).sum()} excluded")

# Simulated predictions and targets
pred_logits = torch.randn(num_nodes, 2)  # model predictions for all nodes
target_labels = torch.randint(0, 2, (num_nodes,))

# Without mask: loss on all nodes
loss_all = F.cross_entropy(pred_logits, target_labels, reduction='sum')

# With mask: loss only on train+val nodes
loss_masked = F.cross_entropy(pred_logits[label_mask], target_labels[label_mask], reduction='sum')

print(f"\nLabel loss (all nodes):       {loss_all.item():.4f}")
print(f"Label loss (masked, 5 nodes): {loss_masked.item():.4f}")
print(f"\nThe masked loss excludes test nodes — their labels cannot influence gradients.")

## 3. Implementation Details

The mask flows through the full training pipeline:

### Pipeline (`bigg/extension/pipeline.py`)
1. Constructs `label_mask = train_masks[:, 0] | val_masks[:, 0]` from the original graph's split 0.
2. If BFS reordering is enabled, the mask is reordered with the same permutation as `node_data` (via the permutation returned by `bfs_reorder`).
3. Passes `label_mask` to both the calibration pass and every training iteration.

### Model (`bigg/extension/customized_models.py`)
Both `BiggWithFeatsAndLabels` and `BiggWithConditionedFeats` accept `label_mask` in `predict_node_feats`:

```python
# When mask is provided, label loss excludes test nodes
if label_mask is not None and not label_mask.all():
    ll_label = -F.cross_entropy(pred_logits[label_mask], target_labels[label_mask], reduction='sum')
else:
    ll_label = -F.cross_entropy(pred_logits, target_labels, reduction='sum')
```

The `label_mask.all()` fast path avoids unnecessary indexing when the mask includes all nodes (i.e., when the flag is disabled).

### Chunked training (`bigg/experiments/train_utils.py`)
`sqrtn_forward_backward` extracts `full_label_mask` alongside `full_node_feats` and slices both with the same `[st_delta:st_delta + cur_num]` indexing per chunk. Node ordering is preserved through the chunking, so positional slicing is correct.

### Tree model (`bigg/model/tree_model.py`)
`forward_train` and `forward` accept `label_mask` as a keyword argument and pass it through to `predict_node_feats`. In the per-node sequential path (`forward`), the mask is sliced per-node as `label_mask[[i]]`.

## 4. Scope and Limitations

### Current scope
- **Task:** Anomaly detection only. The mask targets the label loss, which is the leakage vector for `hidden_labels` evaluation.
- **Split:** Uses split 0 from the original graph's `train_masks`/`val_masks`. The benchmark also defaults to split 0, so the masked training aligns with evaluation.
- **Generator:** BiGG only. CGT's label handling is different and will be addressed separately.

### Limitations

**Teacher forcing still sees test labels.** The LSTM state trajectory is conditioned on all ground-truth labels, including test nodes. This means the hidden states that feed into the label predictor are indirectly influenced by test labels — the predictor just isn't optimised to reproduce them. A stricter approach would replace test labels with a learned default embedding during training, at the cost of degrading hidden state quality.

**Single split.** The mask is fixed to split 0. Benchmarks using other splits (trial_id > 0) will have partial overlap between the masked test set and the actual test set for that split. For a fully rigorous setup, one could mask the union of all test nodes across all splits, but this would leave very few nodes for label training.

**Anomaly class imbalance.** Reddit has 366 anomalies out of 10,984 nodes. With ~70% of nodes in the test set, only ~110 anomalies remain for label training. The label predictor has fewer positive examples to learn from, which may affect the quality of generated anomaly labels.

### Planned extension: edge masking for link prediction
The `hidden_links` task has an analogous leakage: BiGG trains on all edges, but the link prediction benchmark holds out edges for testing. Edge masking would remove test edges from the input graph before training. This is a more invasive change than label masking because it modifies the graph structure that BiGG learns from, not just the loss.

## 5. Usage

```bash
# Via shell script — 14th positional argument
bash scripts/train/train_bigg.sh reddit -1 1 300 0.001 256 0.3 0.0 0 True zscore 0.1,0.1 true true

# Via SLURM — set MASK_TEST_LABELS=true in train_bigg.slurm
MASK_TEST_LABELS=true

# Or directly via the pipeline
python -m bigg.extension.pipeline \
  -data_dir reddit \
  --mask_test_labels \
  --hetero_feat \
  ...
```

The flag appends `_masked` to the save name, producing files like:
```
blksize_-1_b_1_lr_0.001_epochs_300_noise_0.3_ss_0.0_norm_zscore_bfs_lw_0.1_0.1_hetero_masked
```

When the flag is not set (default), all codepaths are identical to the unmasked implementation.